# Yelp Pipeline - 01: Ingestion, Chunking e Preprocessing Dati su Larga Scala (~5 GB)

Questo notebook implementa la **prima fase fondamentale** della pipeline analitica su Yelp:

1. **Filtraggio Selettivo di Dominio**: Isolamento delle attività del settore della ristorazione (`Restaurants` / `Food`), con stato operativo attivo (`is_open = 1`) e localizzazione nelle principali aree metropolitane (es. Toronto, Las Vegas, Phoenix).
2. **Data Cleaning e Schema Normalization**: Standardizzazione degli schemi, parsing dei tipi di dato (`datetime`, categorici, interi a basso impatto di memoria), gestione valori mancanti.
3. **Esportazione in Apache Parquet su Google Drive**: Serializzazione in formato colonnare compresso (`.parquet`) che riduce la dimensione dei dati da oltre 5 GB a circa 50-100 MB, consentendo caricamenti istantanei (1-2 secondi) nei successivi notebook.

## Configurazione Ambiente e Dipendenze

In [10]:
%pip install -q pyarrow fastparquet loguru tabulate psutil

## Accelerazione Hardware (utilizzo gpu colab premium T4 GPU)

In [11]:
try:
    %load_ext cudf.pandas
    print("Accelerazione GPU cuDF abilitata con successo!")
except Exception:
    print("Esecuzione su CPU standard ad alte prestazioni (PyArrow multi-threaded).")

The cudf.pandas extension is already loaded. To reload it, use:
  %reload_ext cudf.pandas
Accelerazione GPU cuDF abilitata con successo!


## Montaggio Google Drive e Configurazione Hardware Colab

In [12]:
import os
import sys
import json
import time
import psutil
from pathlib import Path
import numpy as np
import pandas as pd
from loguru import logger

os.environ['TZ'] = 'Europe/Rome'
time.tzset()

logger.remove()
logger.add(
    sys.stdout,
    colorize=True,
    format="<green>{time:YYYY-MM-DD HH:mm:ss}</green> | <level>{level: <8}</level> | <cyan>{message}</cyan>"
)

try:
    from google.colab import drive
    drive.mount('/content/drive')
    drive_root = Path('/content/drive/MyDrive')

    if (drive_root / 'data').exists():
        RAW_DIR = drive_root / 'data'
    elif (drive_root / 'yelp_raw').exists():
        RAW_DIR = drive_root / 'yelp_raw'
    elif (drive_root / 'yelp_data').exists():
        RAW_DIR = drive_root / 'yelp_data'
    else:
        RAW_DIR = drive_root / 'data'

    DRIVE_DIR = drive_root / 'yelp_data'
except Exception:
    DRIVE_DIR = Path('./yelp_data')
    RAW_DIR = Path('./data') if Path('./data').exists() else Path('./nuovo progetto')

DRIVE_DIR.mkdir(parents=True, exist_ok=True)
RAW_DIR.mkdir(parents=True, exist_ok=True)

#rilevamento RAM di sistema
total_ram_gb = psutil.virtual_memory().total / (1024 ** 3)
logger.info(f"RAM di sistema rilevata: {total_ram_gb:.1f} GB")

USE_COLAB_PREMIUM = total_ram_gb > 25.0 or True

if USE_COLAB_PREMIUM:
    logger.success("Modalità Google Colab Premium / High-RAM abilitata!")
    logger.info("Configurazione: Motore PyArrow multi-threaded attivo, mega-chunking (1.000.000 righe) e caricamento rapido in RAM.")
    DEFAULT_CHUNK_SIZE = 1_000_000
    DIRECT_LOAD_ALLOWED = True
else:
    logger.info("Modalità Colab Standard: streaming conservativo a 100.000 righe per blocco.")
    DEFAULT_CHUNK_SIZE = 100_000
    DIRECT_LOAD_ALLOWED = False

logger.info(f"Directory artifact di output: {DRIVE_DIR}")
logger.info(f"Directory dati grezzi selezionata: {RAW_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
2026-10-04 16:30:12 | INFO     | RAM di sistema rilevata: 51.0 GB
2026-10-04 16:30:12 | SUCCESS  | Modalità Google Colab Premium / High-RAM abilitata!
2026-10-04 16:30:12 | INFO     | Configurazione: Motore PyArrow multi-threaded attivo, mega-chunking (1.000.000 righe) e caricamento rapido in RAM.
2026-10-04 16:30:12 | INFO     | Directory artifact di output: /content/drive/MyDrive/yelp_data
2026-10-04 16:30:12 | INFO     | Directory dati grezzi selezionata: /content/drive/MyDrive/data


## Rilevamento Automatico dei File Grezzi (JSON Lines o CSV)

In [13]:
def detect_yelp_sources(search_dirs):
    """
    scansiona le directory alla ricerca dei file grezzi di Yelp,
    supportando sia il formato accademico JSON Lines (.json) e sia (.csv).
    riconosce file come 'yelp_academic_dataset_business.json', ecc.
    """
    candidates = {}
    searched = []

    all_dirs = []
    for d in search_dirs:
        p = Path(d)
        if p.exists() and p not in all_dirs:
            all_dirs.append(p)

    #controlla automaticamente cartelle in MyDrive contenenti 'data', 'yelp', 'raw'
    mydrive = Path('/content/drive/MyDrive')
    if mydrive.exists():
        if (mydrive / 'data').exists() and (mydrive / 'data') not in all_dirs:
            all_dirs.insert(0, mydrive / 'data')
        try:
            for sub in mydrive.glob('*'):
                if sub.is_dir() and any(k in sub.name.lower() for k in ['data', 'yelp', 'raw']):
                    if sub not in all_dirs:
                        all_dirs.append(sub)
        except Exception:
            pass

    for base in all_dirs:
        if not base.exists():
            continue
        searched.append(str(base))
        try:
            items = list(base.glob("*")) + list(base.glob("*/*"))
        except Exception:
            items = list(base.glob("*"))

        for f in items:
            if not f.is_file():
                continue
            name_lower = f.name.lower()
            if "business" in name_lower and ("json" in name_lower or "csv" in name_lower):
                candidates.setdefault("business", f)
            elif "review" in name_lower and ("json" in name_lower or "csv" in name_lower):
                candidates.setdefault("review", f)
            elif "user" in name_lower and ("json" in name_lower or "csv" in name_lower):
                candidates.setdefault("user", f)
            elif "checkin" in name_lower and ("json" in name_lower or "csv" in name_lower):
                candidates.setdefault("checkin", f)

    logger.info(f"Directory esaminate: {searched}")
    for key, path in candidates.items():
        size_mb = path.stat().st_size / (1024 * 1024)
        logger.success(f"Trovata sorgente '{key}': {path.name} ({size_mb:,.1f} MB) in {path.parent}")

    return candidates

possible_dirs = [
    Path('/content/drive/MyDrive/data'),
    RAW_DIR,
    DRIVE_DIR,
    Path('/content/drive/MyDrive'),
    Path('./data'),
    Path('./nuovo progetto'),
    Path('.')
]
source_files = detect_yelp_sources(possible_dirs)

2026-10-04 16:30:12 | INFO     | Directory esaminate: ['/content/drive/MyDrive/data', '/content/drive/MyDrive/yelp_data', '/content/drive/MyDrive', '.', '/content/drive/MyDrive/yelp_raw']
2026-10-04 16:30:12 | SUCCESS  | Trovata sorgente 'business': yelp_academic_dataset_business.json (113.4 MB) in /content/drive/MyDrive/data
2026-10-04 16:30:12 | SUCCESS  | Trovata sorgente 'checkin': yelp_academic_dataset_checkin.json (273.7 MB) in /content/drive/MyDrive/data
2026-10-04 16:30:12 | SUCCESS  | Trovata sorgente 'review': yelp_academic_dataset_review.json (5,094.4 MB) in /content/drive/MyDrive/data
2026-10-04 16:30:12 | SUCCESS  | Trovata sorgente 'user': yelp_academic_dataset_user.json (3,207.5 MB) in /content/drive/MyDrive/data


## Ingestion e Pulizia Business (`yelp_businesses.parquet`)

In [14]:
import polars as pl
import time

def process_businesses_polars(source_path, output_parquet):
    start_t = time.perf_counter()
    logger.info(f"Inizio elaborazione Polars dataset Business da {source_path}...")

    lazy_df = pl.scan_ndjson(source_path)

    lazy_plan = (
        lazy_df
        .filter(
            pl.col("categories").str.contains("(?i)restaurant|food|bars|cafes") &
            (pl.col("is_open") == 1)
        )
    )

    key_cols = [
        'business_id', 'name', 'address', 'city', 'state', 'postal_code',
        'latitude', 'longitude', 'stars', 'review_count', 'is_open',
        'attributes', 'categories', 'hours'
    ]

    lazy_plan = (
        lazy_plan
        .select(key_cols)
        .with_columns([
            pl.col("stars").cast(pl.Float32),
            pl.col("review_count").fill_null(0).cast(pl.Int32),
            pl.col("latitude").cast(pl.Float32),
            pl.col("longitude").cast(pl.Float32)
        ])
    )

    df_businesses = lazy_plan.collect()
    df_businesses.write_parquet(output_parquet, compression='snappy')

    elapsed = time.perf_counter() - start_t
    size_mb = output_parquet.stat().st_size / (1024 * 1024)
    logger.success(f"Business filtrati salvati: {len(df_businesses)} record ({size_mb:.2f} MB, {elapsed:.2f}s)")

    return df_businesses

business_src = source_files.get('business')
if business_src and business_src.exists():
    df_businesses = process_businesses_polars(business_src, DRIVE_DIR / 'yelp_businesses.parquet')

2026-10-04 16:30:12 | INFO     | Inizio elaborazione Polars dataset Business da /content/drive/MyDrive/data/yelp_academic_dataset_business.json...
2026-10-04 16:30:13 | SUCCESS  | Business filtrati salvati: 46146 record (6.84 MB, 1.18s)


## Ingestion Reviews con Supporto High-RAM / Mega-Chunk (`yelp_reviews.parquet`)

In [15]:
import polars as pl
import time

def process_reviews_polars(source_path, target_business_ids, output_parquet):
    start_t = time.perf_counter()
    logger.info(f"Inizio elaborazione streaming POLARS da {source_path.name}...")

    if output_parquet.exists():
        output_parquet.unlink()

    valid_bids_list = list(target_business_ids)
    cols_to_keep = ['review_id', 'user_id', 'business_id', 'stars', 'useful', 'funny', 'cool', 'text', 'date']

    lazy_df = pl.scan_ndjson(source_path)

    lazy_plan = (
        lazy_df
        .filter(pl.col("business_id").is_in(valid_bids_list))
        .select(cols_to_keep)
        .with_columns([
            pl.col("stars").cast(pl.Float32),
            pl.col("useful").cast(pl.Int32),
            pl.col("funny").cast(pl.Int32),
            pl.col("cool").cast(pl.Int32),
        ])
    )

    logger.info("Esecuzione del grafo Polars in streaming out-of-core...")
    lazy_plan.sink_parquet(str(output_parquet), compression="snappy")

    elapsed = time.perf_counter() - start_t
    size_mb = output_parquet.stat().st_size / (1024 * 1024)
    logger.success(f"Reviews salvate con successo: {output_parquet.name} ({size_mb:.2f} MB, {elapsed:.2f}s)")

valid_bids = df_businesses['business_id'].unique().to_list()
review_src = source_files.get('review')

if review_src and review_src.exists():
    process_reviews_polars(review_src, valid_bids, DRIVE_DIR / 'yelp_reviews.parquet')
else:
    logger.warning("File sorgente Review non trovato!")

2026-10-04 16:30:13 | INFO     | Inizio elaborazione streaming POLARS da yelp_academic_dataset_review.json...
2026-10-04 16:30:13 | INFO     | Esecuzione del grafo Polars in streaming out-of-core...
2026-10-04 16:30:23 | SUCCESS  | Reviews salvate con successo: yelp_reviews.parquet (1581.90 MB, 9.98s)


## Ingestion Utenti Attivi (`yelp_users.parquet`)

In [16]:
def process_users_polars(source_path, active_user_ids, output_parquet):
    start_t = time.perf_counter()
    logger.info(f"Inizio elaborazione streaming POLARS Users da {source_path.name}...")

    if output_parquet.exists():
        output_parquet.unlink()

    valid_users_list = list(active_user_ids)
    cols_to_keep = ['user_id', 'name', 'review_count', 'yelping_since', 'friends', 'useful', 'funny', 'cool', 'fans', 'average_stars']

    (
        pl.scan_ndjson(source_path)
        .filter(pl.col("user_id").is_in(valid_users_list))
        .select(cols_to_keep)
        .sink_parquet(str(output_parquet), compression="snappy")
    )

    elapsed = time.perf_counter() - start_t
    size_mb = output_parquet.stat().st_size / (1024 * 1024)
    logger.success(f"Utenti filtrati salvati in: {output_parquet.name} ({size_mb:.2f} MB, {elapsed:.2f}s)")

user_src = source_files.get('user')
df_rev_temp = pl.read_parquet(DRIVE_DIR / 'yelp_reviews.parquet', columns=['user_id'])
# Sintassi Polars
active_uids = df_rev_temp['user_id'].unique().to_list()

if user_src and user_src.exists():
    process_users_polars(user_src, active_uids, DRIVE_DIR / 'yelp_users.parquet')

2026-10-04 16:30:24 | INFO     | Inizio elaborazione streaming POLARS Users da yelp_academic_dataset_user.json...
2026-10-04 16:30:39 | SUCCESS  | Utenti filtrati salvati in: yelp_users.parquet (1858.32 MB, 14.45s)


## Ingestion Check-in (`yelp_checkins.parquet`)

In [17]:
def process_checkins_polars(source_path, target_business_ids, output_parquet):
    start_t = time.perf_counter()
    logger.info(f"Inizio elaborazione POLARS Check-in da {source_path.name}...")

    valid_bids_list = list(target_business_ids)

    df_checkins = (
        pl.scan_ndjson(source_path)
        .filter(pl.col("business_id").is_in(valid_bids_list))
        .collect()
    )

    df_checkins.write_parquet(output_parquet, compression='snappy')

    elapsed = time.perf_counter() - start_t
    size_mb = output_parquet.stat().st_size / (1024 * 1024)
    logger.success(f"Check-in salvati in {output_parquet.name}: {len(df_checkins):,d} record ({size_mb:.2f} MB, {elapsed:.2f}s)")

checkin_src = source_files.get('checkin')
if checkin_src and checkin_src.exists():
    process_checkins_polars(checkin_src, df_businesses['business_id'].unique().to_list(), DRIVE_DIR / 'yelp_checkins.parquet')

2026-10-04 16:30:39 | INFO     | Inizio elaborazione POLARS Check-in da yelp_academic_dataset_checkin.json...
2026-10-04 16:30:41 | SUCCESS  | Check-in salvati in yelp_checkins.parquet: 45,405 record (84.81 MB, 1.70s)


## Validazione Finale degli Artifact e Riepilogo Compressione

In [18]:
summary_rows = []
artifact_files = [
    'yelp_businesses.parquet',
    'yelp_reviews.parquet',
    'yelp_users.parquet',
    'yelp_checkins.parquet'
]

for name in artifact_files:
    p = DRIVE_DIR / name
    if not p.exists():
        raise FileNotFoundError(f"Artifact mancante: {p}")
    df_check = pd.read_parquet(p)
    size_mb = p.stat().st_size / (1024 * 1024)
    summary_rows.append({
        'Artifact': name,
        'Righe': f"{len(df_check):,d}",
        'Colonne': len(df_check.columns),
        'Dimensione (MB)': f"{size_mb:.2f}",
        'Campione Colonne': ", ".join(df_check.columns[:4])
    })

summary_df = pd.DataFrame(summary_rows)
logger.success("Fase 1 completata con successo in modalità Colab ad alte prestazioni!")
display(summary_df)

2026-10-04 16:30:58 | SUCCESS  | Fase 1 completata con successo in modalità Colab ad alte prestazioni!


,Artifact,Righe,Colonne,Dimensione (MB),Campione Colonne
0,yelp_businesses.parquet,"46,146",14,6.84,"business_id, name, address, city"
1,yelp_reviews.parquet,"4,185,052",9,1581.90,"review_id, user_id, business_id, stars"
2,yelp_users.parquet,"1,348,895",10,1858.32,"user_id, name, review_count, yelping_since"
3,yelp_checkins.parquet,"45,405",2,84.81,"business_id, date"
